# GEANT h96 DG-STMTL-based candidate — verified preparation and separately authorized training

Candidate bundle: `geant_dg_stmtl_h96_colab_v1_review3`. This notebook validates four uploaded artifacts by SHA-256, extracts only the exact source and train/validation inventories into one fresh session, runs synthetic tests and a train/validation preflight, and leaves training disabled. It never selects, opens, or evaluates official test observations. Each future seed requires a separate exact confirmation and a new local/Drive run destination. Colab runtime behavior remains unverified until a human runs this notebook there.

In [ ]:
from pathlib import Path
import hashlib, importlib.metadata, json, os, platform, re, shutil, stat, subprocess, sys, uuid, zipfile, io

SOURCE_ZIP_SHA256 = 'a1b53b60d80776b6ec11baf3f0d91491f177fa083ebe14d651a979dae135fadf'
SOURCE_MANIFEST_SHA256 = 'f5bdf0ef45089942195bd32d1277507023a545aefa129a5b95ee56454f2e67fc'
DATASET_ZIP_SHA256 = 'df2bac044c9b9a577d5117310a99022cce402b9370e2ba483ea18b8d7045521e'
DATASET_MANIFEST_SHA256 = '70f60f09b71c0cb89003ac2987af4433df0eeeb64b696845faa25d0fbadb8a95'
DATASET_METADATA_CANONICAL_SHA256 = '71e949092897d273aee39d2b0893ad48b31861b477808c98379f49190c4612d4'
REQUIREMENTS = {'numpy': '1.26.4', 'torch': '2.12.0', 'psutil': '7.2.2', 'pytest': '9.1.1'}
RESOURCE_POLICY_ID = 'colab-linux-process-memory-v3-visible-cgroup'
PROCESS_PEAK_LIMIT_BYTES = 2 * 1024**3
TRAINING_ENABLED = False  # separate review and explicit authorization required
SEEDS = (42, 123, 2026)
SESSION_ROOT = Path('/content') / f'geant_dg_stmtl_h96_review3_{uuid.uuid4().hex}'
if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f'FAIL environment: expected Python 3.12, got {platform.python_version()}')
installed = {name: importlib.metadata.version(name) for name in REQUIREMENTS}
if installed != REQUIREMENTS:
    raise RuntimeError(f'FAIL dependencies: expected exact pins {REQUIREMENTS}, got {installed}. Install the bundled requirements-colab-h96.txt in a fresh Python 3.12 runtime, restart, then run from the top; this notebook will not silently replace packages.')
if SESSION_ROOT.exists():
    raise FileExistsError(f'Unexpected session collision: {SESSION_ROOT}')
SESSION_ROOT.mkdir(exist_ok=False)
print({'stage': 'environment', 'status': 'PASS', 'python': platform.python_version(), 'packages': installed, 'cpu_only': True, 'training_enabled': TRAINING_ENABLED, 'session': str(SESSION_ROOT)})

## Upload and stage the four pinned files

Upload the source ZIP, source manifest, dataset ZIP, and dataset manifest when prompted. Colab may rename duplicates with `(1)`; files are assigned only by exact SHA-256, not by filename. The fresh UUID session avoids reusing partial extractions. The source ZIP has a 30 MiB uncompressed limit; the dataset archive is checked against its manifest's exact nine train/validation members and byte hashes. No test-split member is accepted.

In [ ]:
from google.colab import files

EXPECTED_UPLOADS = {SOURCE_ZIP_SHA256: 'source_zip', SOURCE_MANIFEST_SHA256: 'source_manifest', DATASET_ZIP_SHA256: 'dataset_zip', DATASET_MANIFEST_SHA256: 'dataset_manifest'}
uploaded = files.upload()
role_bytes = {}
for filename, content in uploaded.items():
    digest = hashlib.sha256(content).hexdigest()
    if digest not in EXPECTED_UPLOADS:
        raise RuntimeError(f'FAIL upload: unexpected file/hash for {filename!r}: {digest}')
    role = EXPECTED_UPLOADS[digest]
    if role in role_bytes:
        raise RuntimeError(f'FAIL upload: duplicate artifact for role {role}')
    role_bytes[role] = content
if set(role_bytes) != {'source_zip', 'source_manifest', 'dataset_zip', 'dataset_manifest'}:
    raise RuntimeError(f'FAIL upload: missing approved files: {sorted(set(EXPECTED_UPLOADS.values()) - set(role_bytes))}')
UPLOAD_ROOT = SESSION_ROOT / 'uploads'
UPLOAD_ROOT.mkdir(exist_ok=False)
for role, content in role_bytes.items():
    with (UPLOAD_ROOT / f'{role}.bin').open('xb') as stream:
        stream.write(content); stream.flush(); os.fsync(stream.fileno())

def sha256_bytes(value):
    return hashlib.sha256(value).hexdigest()

if sha256_bytes(role_bytes['source_manifest']) != SOURCE_MANIFEST_SHA256 or sha256_bytes(role_bytes['source_zip']) != SOURCE_ZIP_SHA256:
    raise RuntimeError('FAIL source package hash pin')
if sha256_bytes(role_bytes['dataset_manifest']) != DATASET_MANIFEST_SHA256 or sha256_bytes(role_bytes['dataset_zip']) != DATASET_ZIP_SHA256:
    raise RuntimeError('FAIL dataset package hash pin')
source_manifest = json.loads(role_bytes['source_manifest'])
dataset_manifest = json.loads(role_bytes['dataset_manifest'])
if source_manifest.get('schema') != 'dg-stmtl-h96-colab-source-v1' or source_manifest.get('archive_name') != 'geant_dg_stmtl_h96_source_v1.zip' or source_manifest.get('archive_sha256') != SOURCE_ZIP_SHA256:
    raise RuntimeError('FAIL source manifest identity/schema')
if dataset_manifest.get('schema') != 'geant-h96-colab-trainval-bundle-v1' or dataset_manifest.get('archive_name') != 'geant_h96_colab_trainval_v1.zip' or dataset_manifest.get('archive_sha256') != DATASET_ZIP_SHA256:
    raise RuntimeError('FAIL dataset manifest identity/schema')
if dataset_manifest.get('exact_archive_members') is None or dataset_manifest.get('topology', {}).get('directed_link_count') != 72 or dataset_manifest.get('window_counts') != {'train': 3924, 'validation': 953}:
    raise RuntimeError('FAIL dataset manifest contract')
DATA_NAMES = {'metadata.json', 'timestamps_train.npy', 'timestamps_validation.npy', 'utilization_train.npy', 'utilization_validation.npy', 'valid_train.npy', 'valid_validation.npy', 'window_index_train_h96.json', 'window_index_validation_h96.json'}
if set(dataset_manifest['exact_archive_members']) != DATA_NAMES or len(dataset_manifest['exact_archive_members']) != len(DATA_NAMES):
    raise RuntimeError('FAIL dataset manifest is not the exact nine-member train/validation allowlist')

def safe_extract(blob, destination, expected_entries, *, max_uncompressed):
    destination = Path(destination)
    if destination.exists() or destination.is_symlink(): raise FileExistsError(destination)
    destination.mkdir(parents=True, exist_ok=False)
    expected = {entry['path']: entry for entry in expected_entries}
    with zipfile.ZipFile(io.BytesIO(blob)) as archive:
        infos = archive.infolist(); names = [item.filename for item in infos]
        if len(names) != len(set(names)) or set(names) != set(expected): raise RuntimeError('FAIL archive member inventory mismatch')
        if sum(item.file_size for item in infos) > max_uncompressed: raise RuntimeError('FAIL archive uncompressed-size limit')
        for info in infos:
            name = info.filename; parts = name.split('/')
            if not name or name.startswith('/') or '\\' in name or ':' in parts[0] or any(part in ('', '.', '..') for part in parts): raise RuntimeError(f'FAIL unsafe archive path: {name!r}')
            mode = (info.external_attr >> 16) & 0xFFFF
            if stat.S_ISLNK(mode) or (mode and not (stat.S_ISREG(mode) or stat.S_ISDIR(mode))): raise RuntimeError(f'FAIL non-regular archive member: {name!r}')
            record = expected[name]
            if info.file_size != record['size_bytes']: raise RuntimeError(f'FAIL archive size mismatch: {name}')
            payload = archive.read(info)
            if sha256_bytes(payload) != record['sha256']: raise RuntimeError(f'FAIL member SHA-256: {name}')
            target = destination.joinpath(*parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            with target.open('xb') as output: output.write(payload)
    return sorted(expected)

PROJECT_ROOT = SESSION_ROOT / 'approved_source_snapshot'
source_names = safe_extract(role_bytes['source_zip'], PROJECT_ROOT, source_manifest['entries'], max_uncompressed=30 * 1024**2)
if 'phase9_geant/dg_stmtl_h96_experiment.py' not in source_names or 'src/gnn/dg_stmtl_link.py' not in source_names or 'requirements-colab-h96.txt' not in source_names or 'data/raw/geant_topology/geant.txt' not in source_names:
    raise RuntimeError('FAIL source archive missing a required runtime/provenance member')
DATA_ROOT = PROJECT_ROOT / 'data/processed/geant_horizon96_streaming_v1'
dataset_names = safe_extract(role_bytes['dataset_zip'], DATA_ROOT, dataset_manifest['artifacts'], max_uncompressed=10 * 1024**2)
if set(dataset_names) != DATA_NAMES or any('test' in Path(name).stem.lower().split('_') for name in dataset_names):
    raise RuntimeError('FAIL extracted dataset contains a non-allowlisted or test-split member')
print({'stage': 'upload_and_archive_integrity', 'status': 'PASS', 'source_members': len(source_names), 'dataset_members': len(dataset_names), 'dataset_scope': 'train/validation only', 'session': str(SESSION_ROOT)})

## Synthetic tests and data preflight

The synthetic tests and model self-check run before loading the staged GEANT train/validation arrays. Then the existing verifier reads only its fixed train/validation allowlist and checks source identities, metadata, split-local windows, 22/36/72 topology structure and directed-edge digest. The dataset sidecar labels itself unapproved; this candidate package does not elevate that self-declared status. A human reviewer must independently accept the pinned artifacts before training authorization.

In [ ]:
for name, module in list(sys.modules.items()):
    if (name == 'phase9_geant' or name.startswith('phase9_geant.') or name == 'src.gnn' or name.startswith('src.gnn.')) and getattr(module, '__file__', None):
        try: Path(module.__file__).resolve().relative_to(PROJECT_ROOT.resolve())
        except ValueError: raise RuntimeError(f'FAIL conflicting preloaded project module: {name} -> {module.__file__}')
if sys.path and sys.path[0] != str(PROJECT_ROOT): sys.path.insert(0, str(PROJECT_ROOT))
test_result = subprocess.run([sys.executable, '-m', 'pytest', '-q', str(PROJECT_ROOT / 'phase9_geant/tests/test_dg_stmtl_h96.py'), '--basetemp', str(SESSION_ROOT / 'pytest-dg')], cwd=PROJECT_ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(test_result.stdout)
if test_result.returncode != 0: raise RuntimeError(f'FAIL synthetic tests with exit {test_result.returncode}')
from phase9_geant import dg_stmtl_h96_experiment as experiment
from phase9_geant import dg_stmtl_h96_workflow as workflow
from phase9_geant import horizon96_streaming_experiment_v2 as base
from phase9_geant.colab_h96_v1 import linux_resources
source_hashes = {entry['path']: entry['sha256'] for entry in source_manifest['entries']}
for name, module in list(sys.modules.items()):
    if (name == 'phase9_geant' or name.startswith('phase9_geant.') or name == 'src.gnn' or name.startswith('src.gnn.')) and getattr(module, '__file__', None):
        module_path = Path(module.__file__).resolve()
        try: relative = module_path.relative_to(PROJECT_ROOT.resolve()).as_posix()
        except ValueError: raise RuntimeError(f'FAIL imported project module escaped bundle: {name} -> {module_path}')
        if relative not in source_hashes or hashlib.sha256(module_path.read_bytes()).hexdigest() != source_hashes[relative]: raise RuntimeError(f'FAIL imported module identity mismatch: {relative}')
resource = linux_resources.preflight(disk_path=SESSION_ROOT)
if resource.get('policy', {}).get('policy_id') != RESOURCE_POLICY_ID or resource['policy'].get('process_peak_limit_bytes') != PROCESS_PEAK_LIMIT_BYTES: raise RuntimeError('FAIL Colab resource policy identity/limit')
if experiment.self_check().get('training') is not False: raise RuntimeError('FAIL self-check unexpectedly enabled training')
verified = experiment._verified_inputs(DATA_ROOT, PROJECT_ROOT)
if experiment._canonical_json_sha256(verified[3]) != DATASET_METADATA_CANONICAL_SHA256: raise RuntimeError('FAIL canonical dataset metadata identity')
if tuple(verified[2]) != ('train', 'validation') or len(verified[2]['train']['windows']) != 3924 or len(verified[2]['validation']['windows']) != 953: raise RuntimeError('FAIL split/window identity')
if len(verified[4]) != 72 or verified[3]['topology']['node_count'] != 22 or verified[3]['topology']['undirected_link_count'] != 36: raise RuntimeError('FAIL ordered topology contract')
if (PROJECT_ROOT / 'data/processed' / experiment.OUTPUT_ROOT_NAME).exists(): raise FileExistsError('Refusing existing local DG output namespace')
from google.colab import drive
drive.mount('/content/drive')
DRIVE_BACKUP_ROOT = Path('/content/drive/MyDrive/SDNResearch/geant_dg_stmtl_h96_v1_review3')
if DRIVE_BACKUP_ROOT.exists() or DRIVE_BACKUP_ROOT.is_symlink(): raise FileExistsError(f'Refusing existing persistent namespace: {DRIVE_BACKUP_ROOT}')
if DRIVE_BACKUP_ROOT.parent.is_symlink() or not DRIVE_BACKUP_ROOT.parent.is_dir() or DRIVE_BACKUP_ROOT.parent.resolve() != DRIVE_BACKUP_ROOT.parent: raise RuntimeError('FAIL expected canonical persistent Drive parent is unavailable')
print({'stage': 'synthetic_and_data_preflight', 'status': 'PASS', 'synthetic_pytest_exit': test_result.returncode, 'policy': RESOURCE_POLICY_ID, 'process_peak_limit_bytes': PROCESS_PEAK_LIMIT_BYTES, 'train_windows': 3924, 'validation_windows': 953, 'history': 672, 'horizon': 96, 'directed_links': 72, 'test_access': False, 'training_enabled': TRAINING_ENABLED, 'backup_root': str(DRIVE_BACKUP_ROOT)})

## Separate training authorization — disabled by default

Do not change `TRAINING_ENABLED` unless the independent review and preflight have been accepted. Each seed asks for the exact phrase `AUTHORIZE DG-STMTL H96 SEED N TOPOLOGY`. Runs are sequential and each completed local run is verified before exclusive Drive backup; the copy is then read back and SHA-256 checked before the next seed. Existing local or Drive destinations fail closed. A failed, partial, or disconnected attempt is never resumed automatically. Source code cannot protect against a privileged local user or arbitrary code executing in this process.

In [ ]:
if TRAINING_ENABLED is not True:
    print('Training disabled; no model was trained.')
else:
    for seed in SEEDS:
        response = input(f'Type exactly {workflow.authorization_phrase(seed)} to authorize only seed {seed}: ')
        workflow.require_authorization(seed, response)
        if seed == SEEDS[0]:
            DRIVE_BACKUP_ROOT.mkdir(parents=False, exist_ok=False)
            if DRIVE_BACKUP_ROOT.is_symlink() or DRIVE_BACKUP_ROOT.resolve() != DRIVE_BACKUP_ROOT: raise RuntimeError('Drive namespace failed post-creation identity check')
        elif DRIVE_BACKUP_ROOT.is_symlink() or not DRIVE_BACKUP_ROOT.is_dir() or DRIVE_BACKUP_ROOT.resolve() != DRIVE_BACKUP_ROOT:
            raise RuntimeError('Persistent Drive namespace changed or disappeared between seed runs')
        print({'event': 'run_start', 'seed': seed, 'adjacency': 'hybrid_topology_dynamic'}, flush=True)
        result = experiment.train_one(DATA_ROOT, repository_root=PROJECT_ROOT, seed=seed, run_training=True, explicit_authorization=True, monitor=linux_resources.LinuxMemoryMonitor())
        local_run = PROJECT_ROOT / 'data/processed' / experiment.OUTPUT_ROOT_NAME / f'dg_stmtl_topology_seed_{seed}'
        checked = workflow.validate_completed_run(local_run, expected_seed=seed)
        backup = workflow.backup_completed_run(local_run, DRIVE_BACKUP_ROOT, expected_seed=seed)
        if backup.get('verified') is not True or backup['artifacts']['best.pt']['sha256'] != checked['checkpoint_sha256']: raise RuntimeError(f'FAIL backup read-back identity for seed {seed}')
        metric = result['validation_metrics']['dg_stmtl_based']
        print({'seed': seed, 'status': result['status'], 'best_epoch': result['best_epoch'], 'MAE': metric['mae'], 'RMSE': metric['rmse'], 'runtime_seconds': result['runtime_seconds'], 'checkpoint_sha256': checked['checkpoint_sha256'], 'Drive_backup_verified': True}, flush=True)
    print('All three separately authorized runs completed; no test evaluation was performed.')

## Comparison after training (validation manifests only)

The existing comparison script accepts exactly nine frozen B-U GNN run manifests and three completed DG run manifests. It checks protocol, seed/mode coverage, 953×72 validation-pair coverage, identical EWMA metrics, common dataset metadata, ordered-edge digest and normalization, then writes a new report directory. This cell is intentionally not auto-run: supply the nine reviewed GNN manifest paths and invoke it only after the candidate training has been separately reviewed. It never loads predictions, checkpoints or arrays, and it does not read the test split.

In [ ]:
# Explicit manifests are required; this report stage performs no implicit file discovery.
GNN_RUN_MANIFESTS = []  # provide exactly nine reviewed run.json paths after all runs are complete
DG_RUN_MANIFESTS = [PROJECT_ROOT / 'data/processed' / experiment.OUTPUT_ROOT_NAME / f'dg_stmtl_topology_seed_{seed}' / 'run.json' for seed in SEEDS]
if GNN_RUN_MANIFESTS:
    if len(GNN_RUN_MANIFESTS) != 9: raise ValueError('Exactly nine GNN manifests are required')
    report_root = SESSION_ROOT / 'comparison_report'
    comparison = subprocess.run([sys.executable, '-m', 'phase9_geant.compare_dg_stmtl_h96', *sum((['--gnn-run-json', str(p)] for p in GNN_RUN_MANIFESTS), []), *sum((['--dg-run-json', str(p)] for p in DG_RUN_MANIFESTS), []), '--output-directory', str(report_root)], cwd=PROJECT_ROOT, text=True, capture_output=True)
    print(comparison.stdout); print(comparison.stderr)
    if comparison.returncode: raise RuntimeError(f'Comparison failed: {comparison.returncode}')
else:
    print('Comparison not run: no GNN manifest paths were supplied.')